# RigGrade preflight on a free GPU (Kaggle or Colab)

Runs the free RigGrade preflight on whatever GPU this notebook has, writes `preflight-report.json`, and prints the card identity the GPU record uses (a hash of the card's UUID and the board serial where the driver exposes it; the UUID itself is never written). Nothing is sent anywhere unless you run the last cell. Not a grade.

Kaggle: Settings, Accelerator, GPU. Colab: Runtime, Change runtime type, GPU.

## 1. Which GPU is this

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version,serial --format=csv

## 2. Install RigGrade from PyPI

In [ ]:
!pip install -q riggrade
import riggrade, runner.preflight as pf
print('riggrade', getattr(riggrade, '__version__', 'installed'))

## 3. Run the preflight

About one to two minutes. A shared or virtualised box (every free notebook GPU is one) is declared with `--shared`, so use outside the slice is not flagged as a foreign process.

In [ ]:
!python -m runner.preflight --shared --seconds 30 --skip-download --dir /tmp

## 4. The record this run would contribute

The identity block below is what the GPU record (docs/gpu-record-method.md, draft) would key on. Serial reads `None` on cards whose driver does not expose it.

In [ ]:
import json
report = json.load(open('preflight-report.json'))
print('rig_id ', report.get('rig_id'))
print('cards  ', json.dumps(report.get('cards'), indent=2))
print('verdict', report.get('verdict'), report.get('problems'))
print('measures', json.dumps({k: v for k, v in report['measures'].items() if not isinstance(v, dict)}, indent=2)[:1500])

## 5. Optional: share the anonymised summary with the pool

`--dry-share` prints exactly what would be sent (no identity, no serial, no hostname). Change it to `--share` to send. The consent text is printed first.

In [ ]:
!python -m runner.preflight --shared --seconds 30 --skip-download --dir /tmp --dry-share